In [ ]:
import pandas as pd
import numpy as np

f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)

In [107]:
W = t_healthy.values
healthy_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        healthy_comp[a,b]=np.sum(W[:,a] > W[:,b])
healthy_comp = healthy_comp / m_t_healthy
print(healthy_comp)

[[0.         0.22368988 0.33725772 ... 0.13639627 0.41005025 0.42857143]
 [0.54515434 0.         0.57501795 ... 0.18880115 0.63460158 0.66245513]
 [0.29605169 0.20904523 0.         ... 0.11500359 0.4011486  0.42038765]
 ...
 [0.82311558 0.77774587 0.83833453 ... 0.         0.89375449 0.90193826]
 [0.19109835 0.14630294 0.20215363 ... 0.04005743 0.         0.25886576]
 [0.10150754 0.06661881 0.10911701 ... 0.03603733 0.15046662 0.        ]]


In [115]:
CONST_k = 0.95
stable_pairs = np.where((healthy_comp > CONST_k) | (healthy_comp < 1-CONST_k), 
                        1, 0)

In [112]:
X = t_cancer.values
cancer_comp = np.zeros((n_mirnas, n_mirnas))
for a in range(n_mirnas):
    for b in range(n_mirnas):
        cancer_comp[a,b]=np.sum(X[:,a] > X[:,b])
cancer_comp = cancer_comp / m_t_cancer
print(cancer_comp)

[[0.    0.45  0.525 ... 0.505 0.67  0.635]
 [0.4   0.    0.505 ... 0.475 0.645 0.665]
 [0.29  0.31  0.    ... 0.38  0.52  0.49 ]
 ...
 [0.4   0.43  0.455 ... 0.    0.59  0.56 ]
 [0.15  0.13  0.18  ... 0.145 0.    0.205]
 [0.14  0.125 0.215 ... 0.23  0.385 0.   ]]


In [ ]:
CONST_l = 0.95
stable_cancer_pairs = np.where((cancer_comp > CONST_l) | (cancer_comp < 1-CONST_l), 
                        1, 0)

In [120]:
identity_matrix = np.identity(n_mirnas,dtype=bool)
candidate_diagnostic_pairs = (((stable_pairs) & (stable_cancer_pairs)) & 
                            ((np.absolute(healthy_comp - stable_cancer_pairs)) > 0.9) &
                            (~identity_matrix))



In [ ]:
candidate_diagnostic_pairs.nonzero()

(array([   0,    0,    0, ..., 2564, 2564, 2564], shape=(897389,)),
 array([ 135,  148,  167, ..., 2557, 2560, 2561], shape=(897389,)))

0.015
